# EDA -- silver

What columns each table in `data/silver/` has and how they relate to each other. No full dumps -- maximum 5 sample rows per table.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


os.chdir(_repo_root(Path.cwd()))

AIRE_PATH = "data/silver/aire.parquet"
ESTACIONES_AIRE_PATH = "data/silver/estaciones_aire.parquet"
TRAFICO_PATH = "data/silver/trafico.parquet"

## `aire`

Bronze -> silver (`unpivot_air_quality`, `src/data/silver/aire.py`):

- Wide -> long pivot: columns `D01..D31`/`V01..V31` (data/validity per day) become rows `fecha`/`dato`/`validez` (`aire.py:33-44`).
- `MAGNITUD` (numeric code, e.g. `8`) -> `magnitud` (label, e.g. `"NO2"`) via the `MAGNITUD_LABELS` dict (`aire.py:15-19,29-31`).
- Days that don't exist in that month's calendar are discarded -- bronze fills them with `dato=0.0`/`validez='N'` instead of NULL, so the filter is on `LAST_DAY`, not nullity (`aire.py:23-28,39-41`).
- Dropped columns: `PROVINCIA`, `MUNICIPIO`, `PUNTO_MUESTREO`, `ANO`, `MES`.

In [ ]:
aire = pd.read_parquet(AIRE_PATH)
aire.shape

In [ ]:
aire.dtypes

In [ ]:
aire.head(5)

In [ ]:
aire.isna().sum()

## `estaciones_aire`

Bronze -> silver (`assign_district`, `src/data/silver/district_join.py`): same columns as bronze plus `COD_DIS` and `NOMBRE`, added by a spatial join (`gpd.sjoin(..., predicate="within")`) between `Point(LONGITUD, LATITUD)` of each station and the geometry of `distritos`. No other column changes.

In [ ]:
estaciones_aire = pd.read_parquet(ESTACIONES_AIRE_PATH)
estaciones_aire.shape

In [ ]:
estaciones_aire.dtypes

In [ ]:
estaciones_aire.head(5)

In [ ]:
estaciones_aire.isna().sum()

## `trafico`

Tabla grande -- se consulta con DuckDB directamente sobre el parquet, sin cargarla entera en memoria.

Bronze -> silver (`clean_traffic`, `src/data/silver/trafico.py`): same schema, column by column, as bronze -- only value-level change: negative sentinel -> `NULL` in `intensidad`/`ocupacion`/`carga`/`vmed` (`trafico.py:18-27`), because negative means "no data" per official CKAN documentation.

In [ ]:
duckdb.sql(f"SELECT count(*) AS n_filas FROM '{TRAFICO_PATH}'")

In [ ]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{TRAFICO_PATH}'")

In [ ]:
duckdb.sql(f"SELECT * FROM '{TRAFICO_PATH}' LIMIT 5")

In [ ]:
duckdb.sql(f"""
    SELECT
        count(*) FILTER (WHERE intensidad IS NULL) AS null_intensidad,
        count(*) FILTER (WHERE ocupacion IS NULL) AS null_ocupacion,
        count(*) FILTER (WHERE carga IS NULL) AS null_carga,
        count(*) FILTER (WHERE vmed IS NULL) AS null_vmed
    FROM '{TRAFICO_PATH}'
""")

## Relationships between tables

- `aire.estacion` <-> `estaciones_aire` (station key): join necessary to have coordinates/district for each air reading.
- `trafico` needs no join with anything: its metadata already carries its own `distrito` (documented in phase 1 -- `spec/features/001-descubrimiento-catalago/findings.md`).

## About `object` dtypes

Many columns come out as `object` in pandas. Not all for the same reason:

1. **Genuine strings** (`magnitud`, `validez`, `tipo_elem`, `error`, `COD_TIPO`, `COD_DIS`...): `object` is correct, not a bug. Candidates for `.astype("category")` if memory optimization is desired (low cardinality), but not urgent.
2. **`aire.fecha` is the odd case**: in the parquet it's a native `DATE` (duckdb confirms it), but `pd.read_parquet` materializes it as `object` (Python `datetime.date` objects) instead of `datetime64[ns]`, because pyarrow converts a `date32` to `object` by default unless explicitly requested via `pd.to_datetime(...)`. Compare with `trafico.fecha` (already `datetime64[us]`, because bronze carries it as timestamp) and with `estaciones_aire["Fecha alta"]` (same `DATE` in bronze, but comes out `datetime64[us]` because that path goes through `duckdb.sql(...).df()` instead of `pd.read_parquet` directly). Recommendation if treating as real date in pandas/Power BI/Gradio: `pd.to_datetime(aire["fecha"])` after load.
3. **Numeric columns never parsed**: `estaciones_aire.COORDENADA_X_ETRS89`/`_Y_ETRS89` (string with comma decimal, e.g. `"439579,3291"`) and `LONGITUD_ETRS89`/`LATITUD_ETRS89` (string in DMS format, e.g. `3°42'43.91"O`) -- redundant with already-clean columns `LONGITUD`/`LATITUD` (float), so no urgency to fix unless used directly.

In [ ]:
aire.dtypes